In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import pickle
from tqdm import tqdm

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
mg_mapping = pd.read_csv('../../SATURN_hypo_september2026_mapping_30_seeds/mouse_ri_30_seeds.csv', index_col = 'barcode')

In [52]:
mo_mapping = pd.read_csv('../../SATURN_hypo_september2026_mapping_30_seeds/vole_ri_30_seeds.csv', index_col = 'barcode')

In [53]:
sam = SAM()
sam.load_data('../../Active_SAM_joined/SAM_RI_joined_soupx_cleaned_01142026.h5ad')

In [54]:
mg_mapping.columns

Index(['seed_0', 'seed_1', 'seed_2', 'seed_3', 'seed_4', 'seed_5', 'seed_6',
       'seed_7', 'seed_8', 'seed_9', 'seed_10', 'seed_11', 'seed_12',
       'seed_13', 'seed_14', 'seed_15', 'seed_16', 'seed_17', 'seed_18',
       'seed_19', 'seed_20', 'seed_21', 'seed_22', 'seed_23', 'seed_24',
       'seed_25', 'seed_26', 'seed_27', 'seed_28', 'seed_29'],
      dtype='object')

In [55]:
mo_mapping

,seed_0,seed_1,seed_2,seed_3,seed_4,seed_5,seed_6,seed_7,seed_8,seed_9,...,seed_20,seed_21,seed_22,seed_23,seed_24,seed_25,seed_26,seed_27,seed_28,seed_29
barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCACAAGCGGAT Run 18 sample 2,089 PVR Six3 Sox3 Gaba,094 SCH Six6 Cdc14a Gaba,099 SBPV-PVa Six6 Satb2 Gaba,099 SBPV-PVa Six6 Satb2 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,099 SBPV-PVa Six6 Satb2 Gaba,094 SCH Six6 Cdc14a Gaba,...,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,089 PVR Six3 Sox3 Gaba,099 SBPV-PVa Six6 Satb2 Gaba,089 PVR Six3 Sox3 Gaba,092 TMv-PMv Tbx3 Hist-Gaba
AAACCCACAGTCTTCC Run 18 sample 2,101 ZI Pax6 Gaba,mo_1,101 ZI Pax6 Gaba,mo_1,mo_49,mo_10,mo_18,103 PVHd-DMH Lhx6 Gaba,mo_1,mo_10,...,mo_3,099 SBPV-PVa Six6 Satb2 Gaba,mo_34,197 SNr Six3 Gaba,mo_5,195 SNr-VTA Pax5 Npas1 Gaba,mo_34,195 SNr-VTA Pax5 Npas1 Gaba,092 TMv-PMv Tbx3 Hist-Gaba,mo_1
AAACCCACATAGACTC Run 18 sample 2,mo_47,143 MM-ant Foxb1 Glut,mo_9,257 SPVC Ccdc172 Glut,159 IF-RL-CLI-PAG Foxa1 Glut,mo_6,mo_2,117 LHA Barhl2 Glut,119 SI-MA-LPO-LHA Skor1 Glut,mo_9,...,mo_2,140 PMd-LHA Foxb1 Glut,119 SI-MA-LPO-LHA Skor1 Glut,159 IF-RL-CLI-PAG Foxa1 Glut,140 PMd-LHA Foxb1 Glut,119 SI-MA-LPO-LHA Skor1 Glut,mo_6,mo_2,141 PH-SUM Foxa1 Glut,mo_9
AAACCCAGTCCCTGTT Run 18 sample 2,140 PMd-LHA Foxb1 Glut,137 PH-an Pitx2 Glut,137 PH-an Pitx2 Glut,mo_14,137 PH-an Pitx2 Glut,159 IF-RL-CLI-PAG Foxa1 Glut,143 MM-ant Foxb1 Glut,140 PMd-LHA Foxb1 Glut,mo_14,141 PH-SUM Foxa1 Glut,...,134 PH-ant-LHA Otp Bsx Glut,mo_14,134 PH-ant-LHA Otp Bsx Glut,mo_14,137 PH-an Pitx2 Glut,137 PH-an Pitx2 Glut,235 PG-TRN-LRN Fat2 Glut,141 PH-SUM Foxa1 Glut,137 PH-an Pitx2 Glut,137 PH-an Pitx2 Glut
AAACCCAGTCTGTAAC Run 18 sample 2,mo_3,102 DMH-LHA Gsx1 Gaba,103 PVHd-DMH Lhx6 Gaba,mo_3,mo_3,097 PVHd-SBPV Six3 Prox1 Gaba,mo_3,mo_3,mo_3,mo_3,...,097 PVHd-SBPV Six3 Prox1 Gaba,097 PVHd-SBPV Six3 Prox1 Gaba,mo_35,mo_3,mo_3,097 PVHd-SBPV Six3 Prox1 Gaba,mo_3,103 PVHd-DMH Lhx6 Gaba,123 DMH Nkx2-4 Glut,mo_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGAGTCGAGGT Run 18 sample 10,061 STR D1 Gaba,061 STR D1 Gaba,081 ACB-BST-FS D1 Gaba,061 STR D1 Gaba,061 STR D1 Gaba,062 STR D2 Gaba,081 ACB-BST-FS D1 Gaba,061 STR D1 Gaba,062 STR D2 Gaba,062 STR D2 Gaba,...,062 STR D2 Gaba,061 STR D1 Gaba,061 STR D1 Gaba,061 STR D1 Gaba,062 STR D2 Gaba,061 STR D1 Gaba,061 STR D1 Gaba,061 STR D1 Gaba,104 TU-ARH Otp Six6 Gaba,062 STR D2 Gaba
TTTGTTGCAGAACCGA Run 18 sample 10,152 RE-Xi Nox4 Glut,144 MM Foxb1 Glut,064 STR-PAL Chst9 Gaba,152 RE-Xi Nox4 Glut,007 L2/3 IT CTX Glut,152 RE-Xi Nox4 Glut,064 STR-PAL Chst9 Gaba,152 RE-Xi Nox4 Glut,152 RE-Xi Nox4 Glut,152 RE-Xi Nox4 Glut,...,152 RE-Xi Nox4 Glut,007 L2/3 IT CTX Glut,mo_38,010 IT AON-TT-DP Glut,007 L2/3 IT CTX Glut,152 RE-Xi Nox4 Glut,152 RE-Xi Nox4 Glut,010 IT AON-TT-DP Glut,136 PMv-TMv Pitx2 Glut,010 IT AON-TT-DP Glut
TTTGTTGCAGCGGATA Run 18 sample 10,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,...,106 PVpo-VMPO-MPN Hmx2 Gaba,mo_26,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba


In [56]:
sam.adata

AnnData object with n_obs × n_vars = 66935 × 23867
    obs: 'n_genes', 'n_counts', 'key', 'leiden_clusters', 'subclass_id_label_mapping', 'subclass_id_label_lc', 'subclass_id_label_mapping_nounlabeled', 'subclass_id_label_crossed', 'subclass_id_label_crossed_nn', 'subclass_lc', 'ss_subclass', 'ss_subclass_nn', 'ss_subclass_nounlabeled_nn', 'ss_subclass_crossed_nn', 'ss_subclass_nounlabeled_nn_agrp', 'yanay_labels', 'yanay_label_nounlabeled'
    var: 'mask_genes', 'means', 'variances', 'weights', 'spatial_dispersions'
    uns: 'dimred_indices', 'path_to_file', 'preprocess_args', 'ranked_genes', 'run_args'
    obsm: 'Raw_X', 'X_pca', 'X_processed', 'X_tsne', 'X_umap'
    varm: 'PCs'
    layers: 'X_disp'
    obsp: 'connectivities', 'distances', 'nnm'

In [59]:
level = 'subclass_lc'

In [60]:
cj_clusters = sam.adata.obs[level].to_frame()
cj_clusters.colums = [level]

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  


In [61]:
cj_clusters

,subclass_lc
AAACCCACAAGCGGAT Run 18 sample 2,17
AAACCCACAGTCTTCC Run 18 sample 2,63
AAACCCACATAGACTC Run 18 sample 2,174
AAACCCAGTCCCTGTT Run 18 sample 2,42
AAACCCAGTCTGTAAC Run 18 sample 2,85
...,...
TTTGTTGAGTCGAGGT Run 18 sample 10,32
TTTGTTGCAGAACCGA Run 18 sample 10,5
TTTGTTGCAGCGGATA Run 18 sample 10,182
TTTGTTGCAGTTGTTG Run 18 sample 10,59


In [62]:
df_comb_mapping = pd.DataFrame(index = cj_clusters.index, columns = ['seed_' + str(i) for i in range(30)])

In [63]:
cj_clusters

,subclass_lc
AAACCCACAAGCGGAT Run 18 sample 2,17
AAACCCACAGTCTTCC Run 18 sample 2,63
AAACCCACATAGACTC Run 18 sample 2,174
AAACCCAGTCCCTGTT Run 18 sample 2,42
AAACCCAGTCTGTAAC Run 18 sample 2,85
...,...
TTTGTTGAGTCGAGGT Run 18 sample 10,32
TTTGTTGCAGAACCGA Run 18 sample 10,5
TTTGTTGCAGCGGATA Run 18 sample 10,182
TTTGTTGCAGTTGTTG Run 18 sample 10,59


In [64]:
for i in tqdm(range(30)):
    mapping_dict = {}
    for lc in range(cj_clusters[level].nunique()):
        mg_mapping_set = mg_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        mo_mapping_set = mo_mapping.loc[cj_clusters[cj_clusters[level] ==lc].index,'seed_' + str(i)]
        if mg_mapping_set.mode()[0] == mo_mapping_set.mode()[0] and len(mg_mapping_set) >25:
            mapping_dict[lc] = mg_mapping_set.mode()[0] 
        else:
            mapping_dict[lc] = 'Unlabeled'
    new_mapping = [mapping_dict[item] for item in cj_clusters[level]]
    df_comb_mapping.loc[:,'seed_' + str(i)] = new_mapping

100%|█████████████████████████████████████████████████████████████████████████| 30/30 [00:14<00:00,  2.11it/s]


In [65]:
df_comb_mapping.to_csv('../../SATURN_hypo_september2026_mapping_30_seeds/mouse_vole_ri_30_seeds.csv')